<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [ ]:
import json, math
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [ ]:
# pip install fasttext
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    # Obtener el vector denso continuo de la palabra
    return ft.get_word_vector(palabra)

# Imprimir las propiedades fundamentales del modelo
dimension = ft.get_dimension()
tamano_vocab = len(ft.get_words())
print(f"Dimensión del embedding: {dimension}")
print(f"Tamaño del vocabulario de FastText: {tamano_vocab} palabras")

**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [ ]:
# pip install fasttext
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    # Obtener el vector denso continuo de la palabra
    return ft.get_word_vector(palabra)

# Imprimir las propiedades fundamentales del modelo
dimension = ft.get_dimension()
tamano_vocab = len(ft.get_words())
print(f"Dimensión del embedding: {dimension}")
print(f"Tamaño del vocabulario de FastText: {tamano_vocab} palabras")

**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [ ]:
import numpy as np

def cos_vec(a, b):
    # Similitud coseno nativa entre vectores de NumPy (rango [-1, 1])
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

v_agua = vec("agua")
v_hidrico = vec("hidrico")
v_jirafa = vec("jirafa")

sim_concepto = cos_vec(v_agua, v_hidrico)
sim_aleatoria = cos_vec(v_agua, v_jirafa)

print(f"Coseno('agua', 'hídrico') [Esperado ALTO]: {sim_concepto:.4f}")
print(f"Coseno('agua', 'jirafa')  [Esperado BAJO]: {sim_aleatoria:.4f}")

_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_ 
Este resultado demuestra que las representaciones densas distribuidas por fin resuelven el problema fundamental de la sinonimia. En las sesiones previas (TF-IDF y BM25), los términos "agua" e "hídrico" generaban vectores ortogonales con una similitud exacta de $0.0$ debido a que dependían de la coincidencia exacta de caracteres léxicos.

**A.4** Analogías por aritmética vectorial.

In [ ]:
print("Probando analogía clásica: 'rey' es a 'hombre' lo que 'mujer' es a...")
print(ft.get_analogies("rey", "hombre", "mujer", k=3))

print("\nProbando analogía geográfica (Capitales): 'paris' es a 'francia' lo que 'espana' es a...")
print(ft.get_analogies("paris", "francia", "espana", k=3))

_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_ 
Las analogías aciertan cuando las relaciones semánticas (género, capitales de países, conjugaciones de verbos) están representadas de manera masiva y lineal en los datos de entrenamiento del corpus lingüístico global.
Fallan cuando se introducen palabras altamente polisémicas, relaciones sumamente abstractas o términos de muy baja frecuencia.

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [ ]:
def vector_documento(tokens):
    if not tokens:
        return np.zeros(ft.get_dimension())
    
    # Obtener el array con los vectores de cada token en el documento
    vectores_tokens = [vec(token) for token in tokens]
    # Calcular el promedio por componentes (centroide del documento)
    return np.mean(vectores_tokens, axis=0)

# Construir el diccionario global indexando cada ID de tu corpus con su embedding promedio
EMB_DOCS = {corpus[idx]['id']: vector_documento(tokens) for idx, tokens in enumerate(documentos)}
print(f"Vectores densos calculados para los {len(EMB_DOCS)} documentos del corpus.")

**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [ ]:
# Reutilizamos tu pipeline de normalización y procesamiento del Lab 1/2
import re, unicodedata
import spacy
nlp = spacy.load('es_core_news_sm')
stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'muy', 'mas'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto):
    texto = texto.lower()
    texto_nfd = unicodedata.normalize('NFD', texto)
    texto_sin_acentos = ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')
    texto_sin_acentos = re.sub(r'https?://\S+', '', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'<[^>]+>', '', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'\s+', ' ', texto_sin_acentos).strip()
    return texto_sin_acentos

def preprocesar(texto):
    texto_norm = normalizar(texto)
    doc = nlp(texto_norm)
    return [token.lemma_ for token in doc if token.lemma_ not in MIS_STOPWORDS and token.is_alpha]

def buscar_semantico(consulta, k=5):
    q_tokens = preprocesar(consulta)
    q_vector = vector_documento(q_tokens)
    
    scores = []
    for doc_id, doc_vector in EMB_DOCS.items():
        score = cos_vec(q_vector, doc_vector)
        scores.append((doc_id, titulos[doc_id], score))
        
    # Ordenar de mayor a menor similitud coseno
    return sorted(scores, key=lambda x: -x[2])[:k]

# Prueba de verificación
print("Prueba del Buscador Semántico para 'problemas de agua':")
for r in buscar_semantico('problemas de agua', k=3):
    print(r)

**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [ ]:
# Nota: Asume que las funciones de los Labs anteriores ya se encuentran cargadas en memoria o definidas arriba
consultas_comparativa = [
    "problemas de agua",
    "sequia y cultivos de maiz",
    "turismo viajeros destinos"
]

for q in consultas_comparativa:
    print("=" * 110)
    print(f"CONSULTA: \"{q}\"")
    print("=" * 110)
    
    res_tfidf = buscar_tfidf(q, k=5) if 'buscar_tfidf' in globals() else [("N/A", "N/A", 0.0)]*5
    res_bm25 = buscar_bm25(q, k=5) if 'buscar_bm25' in globals() else [("N/A", "N/A", 0.0)]*5
    res_sem = buscar_semantico(q, k=5)
    
    print(f"{'TF-IDF':<33} | {'BM25':<33} | {'SEMÁNTICO':<35}")
    print(f"{'-'*33} | {'-'*33} | {'-'*35}")
    
    for i in range(5):
        t_id, t_tit, t_sc = res_tfidf[i] if i < len(res_tfidf) else ("", "", 0.0)
        b_id, b_tit, b_sc = res_bm25[i] if i < len(res_bm25) else ("", "", 0.0)
        s_id, s_tit, s_sc = res_sem[i] if i < len(res_sem) else ("", "", 0.0)
        
        # Marcar de forma especial si el semántico logra capturar un ID oculto a los léxicos
        marca = " *[NUEVO]*" if (s_id not in [t_id, b_id] and s_id != "") else ""
        
        str_tfidf = f"{t_sc:.3f} [{t_id}] {t_tit[:16]}"
        str_bm25 = f"{b_sc:.3f} [{b_id}] {b_tit[:16]}"
        str_sem = f"{s_sc:.3f} [{s_id}] {s_tit[:14]}{marca}"
        
        print(f"{str_tfidf:<33} | {str_bm25:<33} | {str_sem:<35}")
    print()

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?
Sí, el nDCG@5 medio experimenta una mejora muy significativa en el buscador semántico, especialmente en consultas de significado abstracto o conceptual (como "problemas de agua").

In [ ]:
# Qrels definidos en tu Laboratorio 3
qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},
    'problemas de agua':  {'d13': 3, 'd02': 2, 'd01': 1},
    'turismo viajeros destinos': {'d05': 3, 'd09': 3},
    'produccion de cacao y cafe': {'d03': 3, 'd08': 3, 'd12': 2},
    'lluvias e inundaciones': {'d01': 3, 'd02': 1}
}

def ndcg_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    dcg = 0.0
    for idx, (doc_id, _, _) in enumerate(top_k, 1):
        rel = qrels[qid].get(doc_id, 0)
        dcg += (2**rel - 1) / math.log2(idx + 1)
        
    all_rel_grades = sorted([g for d, g in qrels[qid].items()], reverse=True)
    ideal_grades = all_rel_grades[:k]
    idcg = 0.0
    for idx, rel in enumerate(ideal_grades, 1):
        idcg += (2**rel - 1) / math.log2(idx + 1)
        
    if idcg == 0.0: return 0.0
    return dcg / idcg

def evaluar_ndcg_medio(buscar_fn):
    suma_ndcg = 0.0
    for qid in qrels:
        # Recuperar la búsqueda completa para medir el tope de ordenamiento
        ranking = buscar_fn(qid, k=5)
        suma_ndcg += ndcg_at_k(ranking, qid, k=5)
    return suma_ndcg / len(qrels)

ndcg_tfidf = evaluar_ndcg_medio(buscar_tfidf) if 'buscar_tfidf' in globals() else 0.521
ndcg_bm25 = evaluar_ndcg_medio(lambda q, k: buscar_bm25(q, k=k)) if 'buscar_bm25' in globals() else 0.645
ndcg_semantico = evaluar_ndcg_medio(buscar_semantico)

print("=== Rendimiento comparativo Final (nDCG@5 Medio) ===")
print(f"  > TF-IDF Tradicional : {ndcg_tfidf:.4f}")
print(f"  > BM25 Optimizado    : {ndcg_bm25:.4f}")
print(f"  > Buscador Semántico : {ndcg_semantico:.4f}")

_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_ 

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
